# Open-Meteo Weather Forecast
Read the JSON file and create a dataframe with `fecha`, `precipitacion` and `temperatura`.

In [1]:
import json
import pandas as pd

# Read JSON (fix known typo: double dot in a temperature value)
with open('dataset/open-meteo-weather-forecast.json') as f:
    raw = f.read()
raw = raw.replace('..', '.')
data = json.loads(raw)

df = pd.DataFrame({
    'fecha': pd.to_datetime(data['hourly']['time']),
    'precipitacion': data['hourly']['precipitation'],
    'temperatura': data['hourly']['temperature_2m']
})

print(f'Shape: {df.shape}')
df.head()

Shape: (27720, 3)


,fecha,precipitacion,temperatura
0,2023-01-01 00:00:00,0.0,6.6
1,2023-01-01 01:00:00,0.0,6.1
2,2023-01-01 02:00:00,0.0,6.6
3,2023-01-01 03:00:00,0.0,6.3
4,2023-01-01 04:00:00,0.0,6.2


## Verify hourly range 2023-01-01 to 2026-02-28

In [2]:
expected_start = pd.Timestamp('2023-01-01 00:00:00')
expected_end = pd.Timestamp('2026-02-28 23:00:00')

actual_start = df['fecha'].min()
actual_end = df['fecha'].max()

print(f'Actual range:   {actual_start} -> {actual_end}')
print(f'Expected range: {expected_start} -> {expected_end}')

# Build complete hourly index and check coverage
full_index = pd.date_range(start=expected_start, end=expected_end, freq='h')
print(f'\nExpected hours: {len(full_index)}')
print(f'Actual hours:   {len(df)}')

missing = full_index.difference(df['fecha'])
print(f'Missing hours:  {len(missing)}')
if len(missing) > 0:
    print(f'Missing range:  {missing.min()} -> {missing.max()}')

assert actual_start == expected_start, f'Start mismatch: {actual_start}'
assert actual_end == expected_end, f'End mismatch: {actual_end} (expected {expected_end})'

Actual range:   2023-01-01 00:00:00 -> 2026-02-28 23:00:00
Expected range: 2023-01-01 00:00:00 -> 2026-02-28 23:00:00

Expected hours: 27720
Actual hours:   27720
Missing hours:  0


## Verify no nulls and no duplicates

In [3]:
nulls = df.isnull().sum()
print('Null values per column:')
print(nulls)
assert nulls.sum() == 0, f'Found {nulls.sum()} null values'

duplicates = df.duplicated().sum()
print(f'\nDuplicated rows: {duplicates}')
assert duplicates == 0, f'Found {duplicates} duplicated rows'

print('\nAll checks passed.')

Null values per column:
fecha            0
precipitacion    0
temperatura      0
dtype: int64

Duplicated rows: 0

All checks passed.


## Statistics

In [4]:
df.describe()

,fecha,precipitacion,temperatura
count,27720,27720.000000,27720.000000
mean,2024-07-31 11:30:00,0.061540,15.660985
min,2023-01-01 00:00:00,0.000000,-4.500000
25%,2023-10-16 17:45:00,0.000000,8.800000
50%,2024-07-31 11:30:00,0.000000,14.500000
75%,2025-05-16 05:15:00,0.000000,22.000000
max,2026-02-28 23:00:00,15.500000,40.000000
std,NaN,0.368278,8.935544


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27720 entries, 0 to 27719
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   fecha          27720 non-null  datetime64[ns]
 1   precipitacion  27720 non-null  float64       
 2   temperatura    27720 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 649.8 KB


## Save to CSV

In [6]:
df.to_csv('dataset/processed/open-meteo-weather-forecast.csv', index=False)
print('Saved to dataset/processed/open-meteo-weather-forecast.csv')

Saved to dataset/processed/open-meteo-weather-forecast.csv
